# Exploration: a tiny PyTorch regression model

This notebook is the **exploration phase** of this demo. It builds a synthetic dataset,
defines a small model, trains it for a few epochs, and checks a validation loss -- all in a
single interactive process, the way a data scientist normally starts.

**This notebook is deliberately NOT the artifact that gets deployed.** Once the modeling
approach looks right, the exact same dataset/model/training-loop logic is extracted into
[`training/train.py`](../training/train.py) -- a plain, versioned Python script that:

- is built into an immutable container image (`training/Containerfile`),
- runs identically whether launched as one process or as many (via `RANK` / `WORLD_SIZE` /
  `LOCAL_RANK`, using PyTorch's `DistributedDataParallel`),
- is what the `TrainJob` (Kubeflow Trainer) and the AI Pipeline actually run on the cluster.

**Notebook = exploration. `train.py` = the reproducible workload.** The last section of
this notebook demonstrates that concretely by importing straight from `training/train.py`.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. A small synthetic dataset

`y = w·x + b + noise` -- deterministic, instant to generate, and easy to reason about. The
point of this demo is the distributed training *architecture*, not the modeling problem.

In [ ]:
def make_dataset(n_samples: int, n_features: int, seed: int, noise_std: float = 0.1):
    generator = torch.Generator().manual_seed(seed)
    x = torch.randn(n_samples, n_features, generator=generator)
    true_weights = torch.linspace(0.5, 2.0, n_features)
    true_bias = 1.5
    noise = torch.randn(n_samples, generator=generator) * noise_std
    y = (x @ true_weights + true_bias + noise).unsqueeze(1)
    return x, y

N_FEATURES = 8
x_train, y_train = make_dataset(n_samples=2048, n_features=N_FEATURES, seed=42)
x_val, y_val = make_dataset(n_samples=512, n_features=N_FEATURES, seed=43)

print("train:", x_train.shape, y_train.shape)
print("val:  ", x_val.shape, y_val.shape)

## 2. A small PyTorch model

A 3-layer MLP -- small enough to train in seconds on a CPU, which keeps this whole demo
extremely lightweight while still exercising the real distributed-training machinery later.

In [ ]:
class TinyRegressor(nn.Module):
    def __init__(self, n_features: int, hidden_size: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        return self.net(x)

model = TinyRegressor(n_features=N_FEATURES)
print(model)

## 3. Train for a few epochs, tracking loss

This is a single-process training loop -- exactly what you'd write first, before thinking
about distribution at all.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

EPOCHS = 20
history = {"epoch": [], "train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    pred = model(x_train)
    train_loss = loss_fn(pred, y_train)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(x_val), y_val)
    model.train()

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss.item())
    history["val_loss"].append(val_loss.item())

    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch={epoch:2d} train_loss={train_loss.item():.4f} val_loss={val_loss.item():.4f}")

## 4. Validate: plot the loss curve

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["epoch"], history["train_loss"], label="train_loss")
plt.plot(history["epoch"], history["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.title("Exploration run (single process)")
plt.show()

print(f"final train_loss={history['train_loss'][-1]:.4f} final val_loss={history['val_loss'][-1]:.4f}")

## 5. From notebook to reproducible training: `training/train.py`

Everything above -- `make_dataset`, `TinyRegressor`, the training loop, the loss prints --
is **exactly** what lives in [`training/train.py`](../training/train.py), just refactored
into a script with:

- `RANK` / `WORLD_SIZE` / `LOCAL_RANK` detection (so it can run as 1 process here, or as N
  processes across N nodes inside a `TrainJob`, unchanged),
- `torch.distributed` + `DistributedDataParallel` when `WORLD_SIZE > 1`,
- a CLI (`--epochs`, `--lr`, `--batch-size`, ...) instead of notebook cells,
- optional MLflow logging if `MLFLOW_TRACKING_URI` is set.

The cell below imports the real classes straight from `training/train.py` and shows that,
run with no distributed environment variables set (like right now, in this notebook), it
detects a single-process, non-distributed run -- `rank=0/1`. The exact same script, run
inside a `TrainJob` via `torchrun`, will instead detect `rank=0/2`, `rank=1/2`, etc.
This is the pedagogical point of this notebook: **notebook = exploration, `train.py` = the
reproducible workload** that is what actually gets containerized and run on the cluster.

In [ ]:
import sys
sys.path.insert(0, "../training")

from train import SyntheticRegressionDataset, TinyRegressor as ReproducibleTinyRegressor, detect_dist_env

env = detect_dist_env()
print(f"Detected distributed environment (from training/train.py): {env}")

ds = SyntheticRegressionDataset(n_samples=8, n_features=N_FEATURES, seed=42)
print("Same dataset generator, same seed, same values:", ds.x[0])

repro_model = ReproducibleTinyRegressor(n_features=N_FEATURES)
print(repro_model)

## Next steps

- `training/train.py` -- the reproducible script, run standalone with `python training/train.py --epochs 5`
- `training/Containerfile` -- builds it into an immutable image
- `manifests/trainjob-template.yaml` -- runs that image as a real distributed `TrainJob`
- `pipeline/pipeline.py` -- orchestrates prepare-data -> distributed-training -> evaluate-model as a real AI Pipeline

See [`DEMO_GUIDE.md`](../DEMO_GUIDE.md) for the full walkthrough.